In [ ]:
import pandas as pd
import numpy as np
import h3
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from pathlib import Path

from sklearn.svm import SVR, LinearSVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
print("Loading final grid dataset...")
taxi_agg = pd.read_parquet("../data/taxi_agg.parquet")
print(f"Final grid loaded. Shape: {taxi_agg.shape}")
taxi_agg.head()

In [ ]:
# Calculate distance to Chicago Loop (downtown center)
def haversine_distance(lat1, lon1, lat2=41.8781, lon2=-87.6298):
    r = 6371 # earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c

In [ ]:
mask = taxi_agg['PickupLatitude'].isnull() | taxi_agg['PickupLongitude'].isnull()
if mask.any():
    centroids = {cell: h3.cell_to_latlng(cell) for cell in taxi_agg.loc[mask, 'h3_index'].unique()}
    lat_map = {cell: latlng[0] for cell, latlng in centroids.items()}
    lon_map = {cell: latlng[1] for cell, latlng in centroids.items()}
    taxi_agg.loc[mask, 'PickupLatitude'] = taxi_agg.loc[mask, 'h3_index'].map(lat_map)
    taxi_agg.loc[mask, 'PickupLongitude'] = taxi_agg.loc[mask, 'h3_index'].map(lon_map)

taxi_agg['distance_to_loop'] = haversine_distance(taxi_agg['PickupLatitude'], taxi_agg['PickupLongitude'])
print(taxi_agg[['h3_index', 'PickupLatitude', 'PickupLongitude', 'distance_to_loop']].head(2))

In [ ]:
# Calendar features
taxi_agg['month'] = taxi_agg['hour'].dt.month
taxi_agg['day_of_week'] = taxi_agg['hour'].dt.weekday
taxi_agg['hour_of_day'] = taxi_agg['hour'].dt.hour
taxi_agg['is_weekend'] = (taxi_agg['day_of_week'] >= 5).astype(int)

# Cyclic temporal features
taxi_agg['hour_sin'] = np.sin(2 * np.pi * taxi_agg['hour_of_day'] / 24)
taxi_agg['hour_cos'] = np.cos(2 * np.pi * taxi_agg['hour_of_day'] / 24)
taxi_agg['month_sin'] = np.sin(2 * np.pi * taxi_agg['month'] / 12)
taxi_agg['month_cos'] = np.cos(2 * np.pi * taxi_agg['month'] / 12)

# Holidays feature
us_holidays = holidays.USA(years=taxi_agg['hour'].dt.year.unique(), state='IL')

taxi_agg['is_holiday'] = taxi_agg['hour'].dt.date.isin(us_holidays).astype(int)

In [ ]:
pois_cat_wide = pd.read_csv("../data/chicago_pois_category_wide.csv")

In [ ]:
pois_cat_wide.head()

In [ ]:

taxi_agg = taxi_agg.merge(
    pois_cat_wide, 
    left_on='h3_index',
    right_on='h3_index',
    how='left'
)

In [ ]:
# TODO: Add more features for example: distance to nearest airport, public transit stops, etc.
# TODO: Add Weather features (e.g., temperature, precipitation) from the weather dataset, merged on hour and location.

In [ ]:
# List all null values in the dataset
null_counts = taxi_agg.isnull().sum()
print("Null values in each column:")
print(null_counts[null_counts > 0])

taxi_agg['poi_cat_automotive'] = taxi_agg['poi_cat_automotive'].fillna(0).astype(int)
taxi_agg['poi_cat_entertainment'] = taxi_agg['poi_cat_entertainment'].fillna(0).astype(int)
taxi_agg['poi_cat_finance'] = taxi_agg['poi_cat_finance'].fillna(0).astype(int)
taxi_agg['poi_cat_food_drink'] = taxi_agg['poi_cat_food_drink'].fillna(0).astype(int)
taxi_agg['poi_cat_grocery'] = taxi_agg['poi_cat_grocery'].fillna(0).astype(int)
taxi_agg['poi_cat_health'] = taxi_agg['poi_cat_health'].fillna(0).astype(int)
taxi_agg['poi_cat_leisure_sports'] = taxi_agg['poi_cat_leisure_sports'].fillna(0).astype(int)
taxi_agg['poi_cat_lodging'] = taxi_agg['poi_cat_lodging'].fillna(0).astype(int)
taxi_agg['poi_cat_nightlife'] = taxi_agg['poi_cat_nightlife'].fillna(0).astype(int)
taxi_agg['poi_cat_services'] = taxi_agg['poi_cat_services'].fillna(0).astype(int)
taxi_agg['poi_cat_shopping'] = taxi_agg['poi_cat_shopping'].fillna(0).astype(int)
taxi_agg['poi_cat_transport'] = taxi_agg['poi_cat_transport'].fillna(0).astype(int)
taxi_agg['poi_cat_civic_community'] = taxi_agg['poi_cat_civic_community'].fillna(0).astype(int)
taxi_agg['poi_cat_education'] = taxi_agg['poi_cat_education'].fillna(0).astype(int)

null_counts = taxi_agg.isnull().sum()
print("Null values in each column:")
print(null_counts[null_counts > 0])

In [ ]:
# Export aggregated feature dataset for modeling
taxi_agg.to_parquet("../data/chicago_taxi_features.parquet")

In [ ]:
taxi_agg.info()

In [ ]:
taxi_agg.shape

**Validation Strategy**: Use a robust **Temporal Split** (avoiding autokorrelation and data leakage from random splits) to evaluate out-of-sample predictive performance.

In [ ]:
## Perform a temporal split (50/20/30) -> (Train/Validation/Test)
unique_dates = pd.Series(taxi_agg['hour'].dt.date.unique()).sort_values().reset_index(drop=True)
n = len(unique_dates)

train_end = unique_dates.iloc[int(n * 0.50)]
val_end   = unique_dates.iloc[int(n * 0.70)]

print(f"Train:      until {train_end}")
print(f"Validation: until {val_end}")

df_train = taxi_agg[taxi_agg['hour'].dt.date < train_end]
df_val   = taxi_agg[(taxi_agg['hour'].dt.date >= train_end) & (taxi_agg['hour'].dt.date < val_end)]
df_test  = taxi_agg[taxi_agg['hour'].dt.date >= val_end]

print(f"Train: {len(df_train)} rows | Val: {len(df_val)} rows | Test: {len(df_test)} rows")

In [ ]:
## Export the splits to parquets:
df_train.to_parquet("../data/df_train.parquet", index=False)
df_val.to_parquet("../data/df_val.parquet",     index=False)
df_test.to_parquet("../data/df_test.parquet",   index=False)